# 3D Breast Surface Reconstruction (BreastNet3D)
This notebook provides a self-contained PyTorch module that reconstructs 3D breast surface volumes from 5-view thermal image groups. It picks up exactly where the U-Net notebook left off.



In [ ]:
# requirements at top of file:
# torch>=2.11, torchvision, tifffile, opencv-python,
# numpy>=2, scipy, scikit-image, matplotlib, tqdm, pandas

import os
import sys
import argparse
import glob
import json
import math
import cv2
import tifffile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.ndimage
import scipy.spatial

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## Section 0: U-Net Architecture
We initialize the U-Net architecture verbatim so that it strictly matches the structure of the checkpoint `.pth` file trained in the prior notebook.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1,
                 base_channels=64, dropout=0.2):
        super().__init__()
        b = base_channels
        self.pool = nn.MaxPool2d(2)
        self.enc1 = DoubleConv(in_channels, b,     dropout=0.0)
        self.enc2 = DoubleConv(b,    b*2,  dropout=0.0)
        self.enc3 = DoubleConv(b*2,  b*4,  dropout=0.1)
        self.enc4 = DoubleConv(b*4,  b*8,  dropout=0.1)
        self.bottleneck = DoubleConv(b*8, b*16, dropout=dropout)
        self.up4 = nn.ConvTranspose2d(b*16, b*8, 2, stride=2)
        self.dec4 = DoubleConv(b*16, b*8,  dropout=0.1)
        self.up3 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = DoubleConv(b*8, b*4,  dropout=0.1)
        self.up2 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = DoubleConv(b*4, b*2,  dropout=0.0)
        self.up1 = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.dec1 = DoubleConv(b*2, b,    dropout=0.0)
        self.out  = nn.Conv2d(b, out_channels, 1)
    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b),e4],1))
        d3=self.dec3(torch.cat([self.up3(d4),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.out(d1)   # logits, no sigmoid

## Section 1: Patient Grouper
This code recursively walks through the dataset folders, grouping thermal images (`.tiff`) and corresponding U-Net masks (`.png`) by `patient_id` and `label`. Only complete groups featuring precisely 5 views (RL, RO, F, LO, LL) are included for reconstruction.


In [ ]:
@dataclass
class PatientGroup:
    patient_id: str
    label: str
    views: Dict[str, Path]
    masks: Dict[str, Path]

def get_view_key(filename):
    prefix = filename[:10].lower()
    if prefix.startswith("right late"): return "RL"
    if prefix.startswith("right obli"): return "RO"
    if prefix.startswith("frontal"): return "F"
    if prefix.startswith("left obliq"): return "LO"
    if prefix.startswith("left later"): return "LL"
    return None

def build_patient_groups(tiff_base, mask_base) -> List[PatientGroup]:
    tiff_path_base = Path(tiff_base)
    mask_path_base = Path(mask_base)
    patient_dict = {}
    
    for tiff_path in tiff_path_base.rglob("*.tiff"):
        rel_path = tiff_path.relative_to(tiff_path_base)
        parts = rel_path.parts
        if len(parts) < 3:
            continue
        patient_id = parts[0]
        label = parts[1]
        filename = parts[-1]
        
        view_key = get_view_key(filename)
        if not view_key:
            continue
            
        dict_key = (patient_id, label)
        if dict_key not in patient_dict:
            patient_dict[dict_key] = {"views": {}, "masks": {}}
            
        patient_dict[dict_key]["views"][view_key] = tiff_path
        
        mask_dir = mask_path_base / patient_id / label
        if mask_dir.exists():
            for mask_file in mask_dir.iterdir():
                if get_view_key(mask_file.name) == view_key:
                    patient_dict[dict_key]["masks"][view_key] = mask_file
                    break

    complete_groups = []
    incomplete_count = 0
    benign_count = 0
    malignant_count = 0
    
    for (patient_id, label), data in patient_dict.items():
        if len(data["views"]) == 5:
            complete_groups.append(PatientGroup(
                patient_id=patient_id,
                label=label,
                views=data["views"],
                masks=data["masks"]
            ))
            if label.lower() == "benign":
                benign_count += 1
            else:
                malignant_count += 1
        else:
            print(f"Warning: Patient {patient_id} ({label}) is missing views. Found {len(data['views'])}/5.")
            incomplete_count += 1
            
    complete_groups.sort(key=lambda x: x.patient_id)
    
    print(f"Total patients found: {len(patient_dict)}")
    print(f"Complete groups (5 views): {len(complete_groups)}")
    print(f"Incomplete groups skipped: {incomplete_count}")
    print(f"Class distribution: Benign={benign_count}, Malignant={malignant_count}")
    
    return complete_groups

## Section 2: Patient Dataset
Builds the PyTorch `Dataset` that normalizes inputs and performs dynamic U-Net mask generation (via the frozen U-Net inference step) if disk masks are missing.

**Thermal Normalization:**
$$ I_{norm} = \frac{I - I_{min}}{I_{max} - I_{min} + \epsilon} $$


In [ ]:
class PatientDataset(Dataset):
    def __init__(self, groups: List[PatientGroup], unet: UNet, device, img_size=256):
        self.groups = groups
        self.unet = unet
        self.device = device
        self.img_size = img_size
        self.view_order = ["RL", "RO", "F", "LO", "LL"]
        
    def __len__(self):
        return len(self.groups)
        
    def __getitem__(self, idx):
        group = self.groups[idx]
        thermals = []
        masks = []
        
        for view in self.view_order:
            tiff_path = group.views[view]
            raw_img = tifffile.imread(str(tiff_path)).astype(np.float32)
            raw_img = cv2.resize(raw_img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
            img_min, img_max = raw_img.min(), raw_img.max()
            img_norm = (raw_img - img_min) / (img_max - img_min + 1e-8)
            thermals.append(img_norm)
            
            mask_path = group.masks.get(view)
            if mask_path and Path(mask_path).exists():
                arr = np.fromfile(str(mask_path), dtype=np.uint8)
                mask_img = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
                mask_img = cv2.resize(mask_img, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
                mask_norm = (mask_img / 255.0).astype(np.float32)
                mask_norm = (mask_norm > 0.5).astype(np.float32)
            else:
                with torch.no_grad():
                    inp = torch.tensor(img_norm).unsqueeze(0).unsqueeze(0).to(self.device)
                    logits = self.unet(inp)
                    mask_pred = torch.sigmoid(logits).squeeze().cpu().numpy()
                    mask_norm = (mask_pred > 0.5).astype(np.float32)
            
            mask_64 = cv2.resize(mask_norm, (64, 64), interpolation=cv2.INTER_NEAREST)
            masks.append(mask_64)
            
        thermals_5ch = np.stack(thermals, axis=0)
        masks_5ch = np.stack(masks, axis=0)
        
        return {
            "masks_5ch": torch.tensor(masks_5ch, dtype=torch.float32),
            "thermals_5ch": torch.tensor(thermals_5ch, dtype=torch.float32),
            "patient_id": group.patient_id,
            "label": group.label,
            "view_order": self.view_order
        }

## Section 3: 3D Encoder (2D $\rightarrow$ Latent)
This neural network transforms the 5-channel 2D masks into a robust 1,000-dimension latent vector. It utilizes $3\times 3$ convolutional filters through five layers initialized with the He / Kaiming normal distribution.


In [ ]:
def init_weights_he(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose3d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm3d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

class Encoder2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(5, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True))
        self.b2 = nn.Sequential(nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True))
        self.b3 = nn.Sequential(nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(True))
        self.b4 = nn.Sequential(nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.b5 = nn.Sequential(nn.Conv2d(256, 512, 3, stride=2, padding=1), nn.BatchNorm2d(512), nn.ReLU(True))
        self.fc = nn.Linear(512 * 2 * 2, 1000)
        self.apply(init_weights_he)
        
    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

## Section 4: 3D Decoder (Latent $\rightarrow$ Volume)
Generates the 3D probability occupancy volume (a shape of $[1, 64, 64, 64]$ voxel grid) from the latent shape code using transposed convolutions. Uses `Sigmoid` to maintain numerical stability and bound all values properly $P_{voxel} \in (0, 1)$.


In [ ]:
class Decoder3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1000, 512 * 2 * 2 * 2)
        self.b1 = nn.Sequential(nn.ConvTranspose3d(512, 256, 4, stride=2, padding=1), nn.BatchNorm3d(256), nn.ReLU(True))
        self.b2 = nn.Sequential(nn.ConvTranspose3d(256, 128, 4, stride=2, padding=1), nn.BatchNorm3d(128), nn.ReLU(True))
        self.b3 = nn.Sequential(nn.ConvTranspose3d(128, 64, 4, stride=2, padding=1), nn.BatchNorm3d(64), nn.ReLU(True))
        self.b4 = nn.Sequential(nn.ConvTranspose3d(64, 32, 4, stride=2, padding=1), nn.BatchNorm3d(32), nn.ReLU(True))
        self.b5 = nn.Sequential(nn.ConvTranspose3d(32, 1, 4, stride=2, padding=1), nn.Sigmoid())
        self.apply(init_weights_he)
        
    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), 512, 2, 2, 2)
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        return x

## Section 5: Differentiable Visual Hull Renderer
Constructs 2D images out of the 3D volume by mapping projection paths along rotation planes using the Beer-Lambert ray integration formula.

**Rotation Matrix over Y axis:**
$$ R_y(\theta) = \begin{bmatrix} \cos(\theta) & 0 & \sin(\theta) & 0 \\ 0 & 1 & 0 & 0 \\ -\sin(\theta) & 0 & \cos(\theta) & 0 \end{bmatrix} $$

**Visual Ray Integration Formulation:**
$$ \text{Projection}(h, w) = 1 - \exp\left(- \sum_d V_{rot}(d, h, w)\right) $$


In [ ]:
def render_projection(volume, theta_deg):
    B, C, D, H, W = volume.shape
    device = volume.device
    dtype = volume.dtype
    
    if not isinstance(theta_deg, torch.Tensor):
        theta_deg = torch.full((B,), float(theta_deg), device=device, dtype=dtype)
        
    theta_rad = theta_deg * math.pi / 180.0
    
    cos_t = torch.cos(theta_rad)
    sin_t = torch.sin(theta_rad)
    zero = torch.zeros_like(theta_rad)
    one = torch.ones_like(theta_rad)
    
    theta_matrix = torch.stack([
        torch.stack([cos_t, zero, sin_t, zero], dim=-1),
        torch.stack([zero, one, zero, zero], dim=-1),
        torch.stack([-sin_t, zero, cos_t, zero], dim=-1)
    ], dim=-2)
    
    grid = F.affine_grid(theta_matrix, volume.shape, align_corners=False)
    V_rot = F.grid_sample(volume, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
    
    v = V_rot.squeeze(1)
    projection = 1.0 - torch.exp(-v.sum(dim=1, keepdim=True))
    
    return projection

## Section 6: Self-Supervised Training
Utilizes the generated projections versus the true segmentations derived through the masks mapping via a Soft Continuous Dice score.

**Dice Loss Equation:**
$$ \text{Loss}_{Dice}(P, T) = 1 - \frac{2 \sum (P \cdot T)}{\sum P^2 + \sum T^2 + \epsilon} $$

Training samples ranges uniformly randomly across 45-degree angle windows per view per forward pass.


In [ ]:
def dice_loss(pred, target, eps=1e-6):
    num = 2 * (pred * target).sum()
    den = pred.pow(2).sum() + target.pow(2).sum() + eps
    return 1 - num / den

def get_view_window(view_idx):
    windows = [
        (-90.0, -67.5),
        (-67.5, -22.5),
        (-22.5, 22.5),
        (22.5, 67.5),
        (67.5, 90.0)
    ]
    return windows[view_idx]

def hd95(p, t):
    if p.sum() == 0 or t.sum() == 0: return 64.0
    p_edges = p ^ scipy.ndimage.binary_erosion(p)
    t_edges = t ^ scipy.ndimage.binary_erosion(t)
    dt_p = scipy.ndimage.distance_transform_edt(~p_edges)
    dt_t = scipy.ndimage.distance_transform_edt(~t_edges)
    d1 = dt_t[p_edges].max() if p_edges.sum() > 0 else 64.0
    d2 = dt_p[t_edges].max() if t_edges.sum() > 0 else 64.0
    return max(d1, d2)

def train_antigravity(config: dict):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    unet = UNet().to(device)
    unet.load_state_dict(torch.load(config['unet_ckpt'], map_location=device))
    unet.eval()
    for p in unet.parameters(): p.requires_grad = False
    
    groups = build_patient_groups(config['tiff_base'], config['mask_base'])
    
    import random
    rng = random.Random(config['seed'])
    benign = [g for g in groups if g.label.lower() == 'benign']
    malignant = [g for g in groups if g.label.lower() != 'benign']
    rng.shuffle(benign)
    rng.shuffle(malignant)
    
    train_benign = benign[:int(len(benign)*0.78)]
    val_benign = benign[int(len(benign)*0.78):]
    train_mal = malignant[:int(len(malignant)*0.78)]
    val_mal = malignant[int(len(malignant)*0.78):]
    
    train_groups = train_benign + train_mal
    val_groups = val_benign + val_mal
    
    train_dataset = PatientDataset(train_groups, unet, device)
    val_dataset = PatientDataset(val_groups, unet, device)
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
    
    encoder = Encoder2D().to(device)
    decoder = Decoder3D().to(device)
    
    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(decoder.parameters()), 
        lr=config['lr'], betas=config['betas']
    )
    
    scaler = torch.cuda.amp.GradScaler(enabled=config['use_amp'])
    Path(config['ckpt_dir']).mkdir(parents=True, exist_ok=True)
    
    best_val_dice = 0.0
    val_angles = [-90.0, -45.0, 0.0, 45.0, 90.0]
    
    import time
    for epoch in range(1, config['epochs'] + 1):
        start_t = time.time()
        encoder.train()
        decoder.train()
        train_loss = 0.0
        
        for batch in train_loader:
            masks_5ch = batch['masks_5ch'].to(device)
            B = masks_5ch.size(0)
            
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=config['use_amp']):
                latent = encoder(masks_5ch)
                volume = decoder(latent)
                
                loss = 0.0
                for i in range(5):
                    low, high = get_view_window(i)
                    for _ in range(config['n_per_view']):
                        theta = torch.rand(B, device=device) * (high - low) + low
                        proj = render_projection(volume, theta)
                        loss += dice_loss(proj, masks_5ch[:, i:i+1])
                
                loss = loss / (5 * config['n_per_view'])
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        
        if epoch % 10 == 0:
            encoder.eval()
            decoder.eval()
            val_dice_total = 0.0
            val_hd_total = 0.0
            count = 0
            
            with torch.no_grad():
                for batch in val_loader:
                    masks_5ch = batch['masks_5ch'].to(device)
                    B = masks_5ch.size(0)
                    with torch.cuda.amp.autocast(enabled=config['use_amp']):
                        latent = encoder(masks_5ch)
                        volume = decoder(latent)
                        
                        for i in range(5):
                            theta = torch.full((B,), val_angles[i], device=device)
                            proj = render_projection(volume, theta)
                            val_dice_total += (1 - dice_loss(proj, masks_5ch[:, i:i+1])).item()
                            
                            proj_bin = (proj > 0.5).cpu().numpy()
                            mask_bin = (masks_5ch[:, i:i+1] > 0.5).cpu().numpy()
                            
                            for b in range(B):
                                val_hd_total += hd95(proj_bin[b,0], mask_bin[b,0])
                                count += 1
                                
            val_dice = val_dice_total / (len(val_loader)*5)
            val_hd = val_hd_total / max(1, count)
            elapsed = time.time() - start_t
            
            print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Dice: {val_dice:.4f} | Val HD: {val_hd:.4f} | LR: {config['lr']} | Elapsed: {elapsed:.2f}s")
            
            ckpt = {
                'epoch': epoch,
                'encoder_state': encoder.state_dict(),
                'decoder_state': decoder.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_val_dice': max(best_val_dice, val_dice),
                'config': config
            }
            torch.save(ckpt, Path(config['ckpt_dir']) / "antigravity_last.pth")
            
            if val_dice > best_val_dice:
                best_val_dice = val_dice
                torch.save(ckpt, Path(config['ckpt_dir']) / "antigravity_best.pth")
                print(f"  -> New best val dice: {best_val_dice:.4f}")

## Section 7: Inference & Export
At inference, we discover the optimal registration angle ($\theta_{best}$) for each projection by minimizing the Dice loss.
After rendering the shape volume, we physically map real thermographic signals from the 2D views onto the outermost visible voxel surface. To average overlapping sections from multiple views, we construct:

**Averaged Voxel Energy Mapping Formulation:**
$$ V_{thermal}(x,y,z) = \frac{\sum_{i} T_{i}\big(\text{Proj}(R_{y}(-\theta_i) \cdot (x,y,z))\big) \cdot \mathbb{I}_{surface}}{\sum_{i} \mathbb{I}_{surface}} $$


In [ ]:
def run_inference(patient_group, encoder, decoder, unet, device):
    dataset = PatientDataset([patient_group], unet, device)
    item = dataset[0]
    masks_5ch = item['masks_5ch'].unsqueeze(0).to(device)
    thermals_5ch = item['thermals_5ch'].numpy()
    
    with torch.no_grad():
        latent = encoder(masks_5ch)
        volume = decoder(latent)
        
        best_angles = []
        for i in range(5):
            low, high = get_view_window(i)
            angles = torch.arange(math.floor(low), math.ceil(high)+1, dtype=torch.float32, device=device)
            min_loss = float('inf')
            best_a = angles[0].item()
            for a in angles:
                proj = render_projection(volume, a)
                loss = dice_loss(proj, masks_5ch[:, i:i+1]).item()
                if loss < min_loss:
                    min_loss = loss
                    best_a = a.item()
            best_angles.append(best_a)
            
    vol_soft = volume[0,0].cpu().numpy()
    vol_bin = (vol_soft > 0.5).astype(np.uint8)
    
    vol_thermal = torch.zeros((1, 1, 64, 64, 64), device=device)
    vol_counts = torch.zeros((1, 1, 64, 64, 64), device=device)
    
    for i, a in enumerate(best_angles):
        theta_rad = a * math.pi / 180.0
        cos_t = math.cos(theta_rad)
        sin_t = math.sin(theta_rad)
        
        theta_matrix = torch.tensor([[
            [cos_t, 0.0, sin_t, 0.0],
            [0.0,   1.0, 0.0,   0.0],
            [-sin_t,0.0, cos_t, 0.0]
        ]], device=device, dtype=torch.float32)
        
        grid = F.affine_grid(theta_matrix, volume.shape, align_corners=False)
        V_rot = F.grid_sample(volume, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        V_rot_bin = V_rot > 0.5
        
        depth_indices = V_rot_bin[0,0].float().argmax(dim=0)
        has_val = V_rot_bin[0,0].float().max(dim=0).values > 0
        
        T_rot = torch.zeros((1, 1, 64, 64, 64), device=device)
        C_rot = torch.zeros((1, 1, 64, 64, 64), device=device)
        
        thermal_view_t = torch.tensor(thermals_5ch[i], device=device)
        thermal_64 = F.interpolate(thermal_view_t.unsqueeze(0).unsqueeze(0), size=(64, 64), mode='bilinear').squeeze()
        
        for h in range(64):
            for w in range(64):
                if has_val[h, w]:
                    d = depth_indices[h, w].long()
                    T_rot[0, 0, d, h, w] = thermal_64[h, w]
                    C_rot[0, 0, d, h, w] = 1.0
                    
        inv_theta_matrix = torch.tensor([[
            [cos_t, 0.0, -sin_t, 0.0],
            [0.0,   1.0, 0.0,    0.0],
            [sin_t, 0.0, cos_t,  0.0]
        ]], device=device, dtype=torch.float32)
        inv_grid = F.affine_grid(inv_theta_matrix, volume.shape, align_corners=False)
        
        T_orig = F.grid_sample(T_rot, inv_grid, mode='nearest', padding_mode='zeros', align_corners=False)
        C_orig = F.grid_sample(C_rot, inv_grid, mode='nearest', padding_mode='zeros', align_corners=False)
        
        vol_thermal += T_orig
        vol_counts += C_orig

    valid = vol_counts > 0
    vol_thermal[valid] /= vol_counts[valid]
    vol_thermal_np = vol_thermal[0,0].cpu().numpy()
    
    out_dict = {
        "volume_binary": vol_bin,
        "volume_soft": vol_soft,
        "volume_thermal": vol_thermal_np,
        "estimated_angles": {k: v for k, v in zip(["RL", "RO", "F", "LO", "LL"], best_angles)},
        "patient_id": patient_group.patient_id,
        "masks_5ch": masks_5ch[0].cpu().numpy(),
        "proj_5ch": [render_projection(volume, a)[0,0].cpu().numpy() for a in best_angles]
    }
    return out_dict

def save_projection_check(out_dict, out_path):
    masks = out_dict["masks_5ch"]
    projs = out_dict["proj_5ch"]
    
    fig, axes = plt.subplots(5, 3, figsize=(9, 15))
    views = ["RL", "RO", "F", "LO", "LL"]
    for i in range(5):
        mask = masks[i]
        proj = (projs[i] > 0.5).astype(np.float32)
        diff = np.abs(mask - proj)
        
        axes[i, 0].imshow(mask, cmap='gray')
        axes[i, 0].set_title(f"{views[i]} Mask")
        axes[i, 1].imshow(proj, cmap='gray')
        axes[i, 1].set_title(f"{views[i]} Proj")
        axes[i, 2].imshow(diff, cmap='hot')
        axes[i, 2].set_title(f"{views[i]} Diff")
        
        for ax in axes[i]: ax.axis('off')
        
    plt.tight_layout()
    from io import BytesIO
    buf = BytesIO()
    plt.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
    img = cv2.imdecode(arr, 1)
    is_success, buffer = cv2.imencode(".png", img)
    if is_success:
        buffer.tofile(str(out_path))

## Section 8: Asymmetry Features
This phase conducts purely mathematical 3D geometry tests against the final output volumes, measuring spatial boundaries and calculating statistical left/right asymmetry thresholds which directly impact neural classification phases later.

**Volume Asymmetry:**
$$ Asymmetry_{LR} = \frac{\mid V_{left} - V_{right} \mid}{V_{left} + V_{right} + \epsilon} $$


In [ ]:
def extract_asymmetry_features(volume_binary, volume_thermal) -> dict:
    vol_voxels = int(np.sum(volume_binary))
    z, y, x = np.nonzero(volume_binary)
    if len(x) > 0:
        cx, cy, cz = float(np.mean(x)), float(np.mean(y)), float(np.mean(z))
        min_x, max_x = x.min(), x.max()
        min_y, max_y = y.min(), y.max()
        min_z, max_z = z.min(), z.max()
        bbox_w = int(max_x - min_x + 1)
        bbox_h = int(max_y - min_y + 1)
        bbox_d = int(max_z - min_z + 1)
    else:
        cx, cy, cz = 0.0, 0.0, 0.0
        bbox_w, bbox_h, bbox_d = 0, 0, 0
        
    eroded = scipy.ndimage.binary_erosion(volume_binary)
    surface = (volume_binary > 0) & (~eroded)
    surface_voxels = int(np.sum(surface))
    
    left_half = volume_binary[:, :, :32]
    right_half = volume_binary[:, :, 32:]
    left_half_vol = int(np.sum(left_half))
    right_half_vol = int(np.sum(right_half))
    
    lr_asym = abs(left_half_vol - right_half_vol) / (left_half_vol + right_half_vol + 1e-6)
    
    surface_thermal = volume_thermal[surface]
    
    left_surface = surface.copy()
    left_surface[:, :, 32:] = False
    right_surface = surface.copy()
    right_surface[:, :, :32] = False
    
    temp_left = volume_thermal[left_surface]
    temp_right = volume_thermal[right_surface]
    
    mean_temp_left = float(np.mean(temp_left)) if len(temp_left) > 0 else 0.0
    mean_temp_right = float(np.mean(temp_right)) if len(temp_right) > 0 else 0.0
    thermal_asym = abs(mean_temp_left - mean_temp_right)
    
    std_temp = float(np.std(surface_thermal)) if len(surface_thermal) > 0 else 0.0
    max_temp = float(np.max(surface_thermal)) if len(surface_thermal) > 0 else 0.0
    
    return {
        "volume_voxels": vol_voxels,
        "centroid_x": cx,
        "centroid_y": cy,
        "centroid_z": cz,
        "bbox_w": bbox_w,
        "bbox_h": bbox_h,
        "bbox_d": bbox_d,
        "surface_voxels": surface_voxels,
        "left_half_vol": left_half_vol,
        "right_half_vol": right_half_vol,
        "lr_asymmetry": float(lr_asym),
        "mean_temp_left": mean_temp_left,
        "mean_temp_right": mean_temp_right,
        "thermal_asymmetry": thermal_asym,
        "std_temp": std_temp,
        "max_temp": max_temp
    }

## Section 9: Execution Entry Point
Execution block that binds all argparse configurations and controls workflow routing for notebook and command line execution contexts.


In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("command", choices=["train", "infer", "export", "all"])
    parser.add_argument("--tiff_base", type=str, default=r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\organized_by_patient")
    parser.add_argument("--mask_base", type=str, default=r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\GroundTruth_Masks")
    parser.add_argument("--out_3d_dir", type=str, default=r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\Volumes_3D")
    parser.add_argument("--unet_ckpt", type=str, default="breast_segmentation_unet_gpu.pth")
    parser.add_argument("--ckpt_3d", type=str, default="checkpoints_3d/antigravity_best.pth")
    
    # Simple hack to allow running within Jupyter if sys.argv only has the notebook name
    if 'ipykernel' in sys.modules and len(sys.argv) >= 2 and sys.argv[1].startswith('-f'):
        # Default behavior in a jupyter notebook if not properly passed
        args = parser.parse_args(["all"]) 
    else:
        args = parser.parse_args()
    
    config = {
        'epochs': 400,
        'batch_size': 4,
        'lr': 0.025,
        'betas': (0.5, 0.9),
        'n_per_view': 2,
        'seed': 42,
        'ckpt_dir': "checkpoints_3d",
        'use_amp': torch.cuda.is_available(),
        'tiff_base': args.tiff_base,
        'mask_base': args.mask_base,
        'unet_ckpt': args.unet_ckpt
    }
    
    if args.command in ["train", "all"]:
        train_antigravity(config)
        
    if args.command in ["infer", "all"]:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        unet = UNet().to(device)
        unet.load_state_dict(torch.load(args.unet_ckpt, map_location=device))
        unet.eval()
        
        encoder = Encoder2D().to(device)
        decoder = Decoder3D().to(device)
        ckpt = torch.load(args.ckpt_3d, map_location=device)
        encoder.load_state_dict(ckpt['encoder_state'])
        decoder.load_state_dict(ckpt['decoder_state'])
        encoder.eval()
        decoder.eval()
        
        groups = build_patient_groups(args.tiff_base, args.mask_base)
        out_dir = Path(args.out_3d_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        
        for g in groups:
            print(f"Running inference for {g.patient_id}...")
            out = run_inference(g, encoder, decoder, unet, device)
            
            p_dir = out_dir / g.patient_id
            p_dir.mkdir(parents=True, exist_ok=True)
            
            np.save(str(p_dir / "volume_binary.npy"), out["volume_binary"])
            np.save(str(p_dir / "volume_thermal.npy"), out["volume_thermal"])
            
            with open(p_dir / "estimated_angles.json", "w") as f:
                json.dump(out["estimated_angles"], f, indent=2)
                
            save_projection_check(out, p_dir / "projection_check.png")
            
    if args.command in ["export", "all"]:
        groups = build_patient_groups(args.tiff_base, args.mask_base)
        out_dir = Path(args.out_3d_dir)
        features_list = []
        
        for g in groups:
            p_dir = out_dir / g.patient_id
            bin_path = p_dir / "volume_binary.npy"
            therm_path = p_dir / "volume_thermal.npy"
            
            if bin_path.exists() and therm_path.exists():
                vol_bin = np.load(str(bin_path))
                vol_therm = np.load(str(therm_path))
                feats = extract_asymmetry_features(vol_bin, vol_therm)
                feats["patient_id"] = g.patient_id
                feats["label"] = g.label
                features_list.append(feats)
                
        if features_list:
            df = pd.DataFrame(features_list)
            cols = ["patient_id", "label"] + [c for c in df.columns if c not in ["patient_id", "label"]]
            df = df[cols]
            df.to_csv(out_dir / "asymmetry_features.csv", index=False)
            print(f"Exported features to {out_dir / 'asymmetry_features.csv'}")

if __name__ == "__main__":
    # main()
    pass